In [1]:
import warnings

import numpy as np
import pandas as pd
import pyreadstat

from argus.utils.PathManager import PathManager_RES_Folder_UPOD

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

# Load data

In [2]:
# First load results and patient selection from the silent trial
path_manager = PathManager_RES_Folder_UPOD()

filename = "prediction_results.parquet"
path_results = path_manager.get_file_path("ARGUS25", "Silent_trial_data", filename)

df = pd.read_parquet(path_results)

In [3]:
df_filter = df.loc[df["missing_fraction"] <= 0.8]
print(
    "Number of unique patients with CAD related symptoms, physical first consultation and less than 80% missingness:"
    f"{df_filter['patientPseuID'].nunique()}"
)
print(df_filter.shape)
print(df_filter.drop_duplicates(subset=["patientPseuID"]).groupby("gender")["Age"].describe())
print(df_filter["BMI"].notna().sum())

Number of unique patients with CAD related symptoms, physical first consultation and less than 80% missingness:978
(985, 43)
        count       mean        std   min   25%   50%   75%   max
gender                                                           
0       488.0  60.518443  16.600131  18.0  53.0  62.5  73.0  92.0
1       490.0  58.836735  16.984231  18.0  50.0  62.0  72.0  92.0
294


# Load OR interventions

### Planned OR interventions

In [4]:
# geplande OK verrichtingen
path_manager = PathManager_RES_Folder_UPOD()

path_plan_verr_ok = path_manager.get_file_path("ARGUS25", "20260803", "geplandeoks_20260803.sas7bdat")

# load verr_ok
df_plan_verr_ok = pyreadstat.read_sas7bdat(path_plan_verr_ok)[0]
df_plan_verr_ok["studyId_1011"] = df_plan_verr_ok["studyId_1011"].astype(int).astype(str)
df_plan_verr_ok["index_dt"] = df_plan_verr_ok["Index_dt"].copy()
df_plan_verr_ok["Index_dt"] = df_plan_verr_ok["index_dt"].dt.date  # make sure to merge on date

In [5]:
# merge with df_patsel
df_plan_verr_ok_pob = df_plan_verr_ok.merge(
    df_filter[["patientPseuID", "Index_dt", "predicted_risk", "missing_fraction"]],
    on=["patientPseuID", "Index_dt"],
    how="inner",
)

In [6]:
# info toevoegen aan df_filter
df_filter["Order_Ok"] = np.nan

df_filter["Order_Ok"] = np.where(
    df_filter.set_index(["patientPseuID", "Index_dt"]).index.isin(
        df_plan_verr_ok_pob.set_index(["patientPseuID", "Index_dt"]).index
    ),
    1,
    df_filter["Order_Ok"],
)

In [7]:
# add a column with the actual planned interventions, and merge per patient visit
reason_lookup = (
    df_plan_verr_ok_pob.dropna(subset=["reason_text"])  # exclude missing reason_text before grouping
    .groupby(["patientPseuID", "Index_dt"])["reason_text"]
    .apply(list)
    .reset_index()
    .rename(columns={"reason_text": "Order_Ok_info"})
)

before = df_filter.shape[0]
df_filter = df_filter.merge(reason_lookup, on=["patientPseuID", "Index_dt"], how="left")
print(f"Rows before: {before}, after: {df_filter.shape[0]}")
assert df_filter.shape[0] == before, "Merge changed row count — check for duplicate keys in reason_lookup"


Rows before: 985, after: 985


Van de geincludeerde consulten zijn er 3 die een verrichting (CAG) op de wachtlijst hebben staan

### Verrichtte OK

In [8]:
# OK verrichtingen
path_manager = PathManager_RES_Folder_UPOD()

path_verr_ok = path_manager.get_file_path("ARGUS25", "20260803", "verr_ok_20260803.sas7bdat")

# load verr_ok
df_verr_ok = pyreadstat.read_sas7bdat(path_verr_ok)[0]
df_verr_ok["studyId_1011"] = df_verr_ok["studyId_1011"].astype(int).astype(str)
df_verr_ok["Index_dt"] = df_verr_ok["index_dt"].dt.date  # make sure to merge on date

In [9]:
# Drop geannuleerde verrichtingen
df_verr_ok = df_verr_ok.loc[df_verr_ok["PRC_Status_display_orginal"] != "Geannuleerd"]

In [10]:
# merge with df_patsel
df_verr_ok_pob = df_verr_ok.merge(df_filter, how="inner")

cols_to_check = [col for col in df_filter.columns if "_info" not in col]

df_verr_ok_pob = df_verr_ok_pob.drop_duplicates(subset=cols_to_check)

print(
    "Number of subjects who had a cardiologic intervention within a year after consulation: "
    f"{df_verr_ok_pob['patientPseuID'].nunique()}"
)
print(
    "Number of records with a cardiologic intervention within a year after consulation: "
    f"{df_verr_ok_pob[['patientPseuID', 'index_dt']].nunique()}"
)


Number of subjects who had a cardiologic intervention within a year after consulation: 111
Number of records with a cardiologic intervention within a year after consulation: patientPseuID    111
index_dt         113
dtype: int64


In [11]:
# get all unique studyId and index_dt combinations in df_verr_ok_pob
unique_combinations = df_verr_ok_pob[
    ["patientPseuID", "index_dt", "cluster_start", "CODE_CODING_DISPLAY", "PRC_Status_display_orginal"]
].drop_duplicates()
print(f"Total unique combinations: {len(unique_combinations)}")
print(
    "Unique interventions per patient per index_dt: "
    f"{unique_combinations[['patientPseuID', 'index_dt']].drop_duplicates().shape[0]}"
)

unique_combinations["verr_ok"] = 1

Total unique combinations: 113
Unique interventions per patient per index_dt: 113


In [12]:
df_filter["Verrichting_Ok"] = np.nan

df_filter["Verrichting_Ok"] = np.where(
    df_filter.set_index(["patientPseuID", "Index_dt"]).index.isin(
        df_verr_ok_pob.set_index(["patientPseuID", "Index_dt"]).index
    ),
    1,
    df_filter["Verrichting_Ok"],
)

In [13]:
# add info
ok_lookup = (
    df_verr_ok_pob.dropna(subset=["CODE_CODING_DISPLAY"])  # exclude missing before grouping
    .groupby(["patientPseuID", "Index_dt"])["CODE_CODING_DISPLAY"]
    .apply(list)
    .reset_index()
    .rename(columns={"CODE_CODING_DISPLAY": "Verr_Ok_info"})
)

before = df_filter.shape[0]
df_filter = df_filter.merge(ok_lookup, on=["patientPseuID", "Index_dt"], how="left")
print(f"Rows before: {before}, after: {df_filter.shape[0]}")
assert df_filter.shape[0] == before, "Merge changed row count — check for duplicate keys in ok_lookup"


Rows before: 985, after: 985


## RAD verrichtingen:

In [14]:
path_verr_rad = path_manager.get_file_path("ARGUS25", "20260803", "rad_verrichting_20260803.sas7bdat")

In [15]:
df_verr_rad = pyreadstat.read_sas7bdat(path_verr_rad)[0]
df_verr_rad["studyId_1011"] = df_verr_rad["studyId_1011"].astype(int).astype(str)
df_verr_rad["Index_dt"] = df_verr_rad["index_dt"].dt.date

In [16]:
# Drop rows where feature is empty
df_verr_rad = df_verr_rad[df_verr_rad["feature"] != ""]

In [17]:
# verr_list = ["CT HART - CORONAIREN",
#              "MRI HART + ADENOSINE STRESS",
#              "PET/CT MYOCARD STRESS",
#              "MYOCARD SPECT/CT STRESS",
#              "MRI HART + REGADENASON STRESS",
#              "CT HART - FAST TRACK",
#              "CT HART - BLANCO"]

In [18]:
mask = df_verr_rad["code_coding_display"].str.contains("echo|Aanvullend|bedthorax", case=False, na=False) | (
    df_verr_rad["code_coding_display"] == "THORAX"
)
df_verr_rad = df_verr_rad[~mask]
verr_list = df_verr_rad["code_coding_display"].unique()

In [19]:
# merge with df_patsel
df_verr_rad_pob = df_verr_rad.merge(
    df_filter[["patientPseuID", "Index_dt"]],
    how="inner",
)

In [20]:
# get unique studyId and index_dt combinations in df_verr_rad
unique_combinations_rad = df_verr_rad_pob[
    ["patientPseuID", "index_dt", "Index_dt", "performedPeriod_start", "code_coding_display"]
].drop_duplicates()
print(f"Total unique combinations in verr_rad: {len(unique_combinations_rad)}")

unique_combinations_rad["verr_rad"] = 1

Total unique combinations in verr_rad: 421


In [21]:
mask = df_verr_rad_pob["code_coding_display"].str.contains("MRI|MRA|PET|SPECT", case=False, na=False)
df_verr_rad_MRI_PET_SPECT = df_verr_rad_pob[mask]
df_verr_rad_CT = df_verr_rad_pob[~mask]

In [22]:
df_filter["Ver_Rad_MRI_PET_SPECT"] = np.nan
df_filter["Ver_Rad_MRI_PET_SPECT"] = np.where(
    df_filter.set_index(["patientPseuID", "Index_dt"]).index.isin(
        df_verr_rad_MRI_PET_SPECT.set_index(["patientPseuID", "Index_dt"]).index
    ),
    1,
    df_filter["Ver_Rad_MRI_PET_SPECT"],
)

print(
    "Unique patients with MRI_PET_SPECT: "
    f"{df_filter['patientPseuID'].loc[df_filter['Ver_Rad_MRI_PET_SPECT'] == 1].nunique()}"
)

Unique patients with MRI_PET_SPECT: 168


In [23]:
# add info
MRI_lookup = (
    df_verr_rad_MRI_PET_SPECT.dropna(subset=["code_coding_display"])  # exclude missing before grouping
    .groupby(["patientPseuID", "Index_dt"])["code_coding_display"]
    .apply(list)
    .reset_index()
    .rename(columns={"code_coding_display": "Ver_Rad_MRI_info"})
)

before = df_filter.shape[0]
df_filter = df_filter.merge(MRI_lookup, on=["patientPseuID", "Index_dt"], how="left")
print(f"Rows before: {before}, after: {df_filter.shape[0]}")
assert df_filter.shape[0] == before, "Merge changed row count — check for duplicate keys in ok_lookup"


Rows before: 985, after: 985


In [24]:
df_filter["Ver_Rad_CT"] = np.nan
df_filter["Ver_Rad_CT"] = np.where(
    df_filter.set_index(["patientPseuID", "Index_dt"]).index.isin(
        df_verr_rad_CT.set_index(["patientPseuID", "Index_dt"]).index
    ),
    1,
    df_filter["Ver_Rad_CT"],
)

print(f"Unique patients with CT: {df_filter['patientPseuID'].loc[df_filter['Ver_Rad_CT'] == 1].nunique()}")

Unique patients with CT: 173


In [25]:
# add info
CT_lookup = (
    df_verr_rad_CT.dropna(subset=["code_coding_display"])  # exclude missing before grouping
    .groupby(["patientPseuID", "Index_dt"])["code_coding_display"]
    .apply(list)
    .reset_index()
    .rename(columns={"code_coding_display": "Ver_Rad_CT_info"})
)

before = df_filter.shape[0]
df_filter = df_filter.merge(CT_lookup, on=["patientPseuID", "Index_dt"], how="left")
print(f"Rows before: {before}, after: {df_filter.shape[0]}")
assert df_filter.shape[0] == before, "Merge changed row count — check for duplicate keys in ok_lookup"

Rows before: 985, after: 985


In [26]:
print(
    "Number of unique patients with CAD related symptoms, physical first consultation and less than 80% missingness: "
    f"{df_filter['patientPseuID'].nunique()}"
)
print(df_filter.shape)

Number of unique patients with CAD related symptoms, physical first consultation and less than 80% missingness: 978
(985, 51)


## Check if radiology rapport is available for all CTs

In [27]:
CT_pat_list = df_filter[["patientPseuID", "Index_dt"]].loc[df_filter["Ver_Rad_CT"] == 1]

In [28]:
## Load radio rapports:
path_manager = PathManager_RES_Folder_UPOD()

path_rad_v = path_manager.get_file_path("ARGUS25", "20260803", "rad_verslagen_20260803.sas7bdat")

# load verr_ok
df_verr_rad_v = pyreadstat.read_sas7bdat(path_rad_v, encoding="latin1")[0]
df_verr_rad_v["studyId_1011"] = df_verr_rad_v["studyId_1011"].astype(int).astype(str)
df_verr_rad_v["Index_dt"] = df_verr_rad_v["index_dt"].dt.date  # make sure to merge on date

In [29]:
# Keep only CT rapports - merge with verr_list
mask = df_verr_rad_v["CODE_CODING_DISPLAY"].isin(verr_list)
df_verr_rad_v = df_verr_rad_v[mask]

# drop MRI/PET/SPECT - keep only CT rapports
mask = df_verr_rad_v["CODE_CODING_DISPLAY"].str.contains("MRI|MRA|PET|SPECT", case=False, na=False)
df_verr_rad_v = df_verr_rad_v[~mask]

In [30]:
df_filter["CT_rapport"] = np.nan
df_filter["CT_rapport"] = np.where(
    df_filter.set_index(["patientPseuID", "Index_dt"]).index.isin(
        df_verr_rad_v.set_index(["patientPseuID", "Index_dt"]).index
    ),
    1,
    df_filter["CT_rapport"],
)

print(f"Unique patients with CT rapport: {df_filter['patientPseuID'].loc[df_filter['CT_rapport'] == 1].nunique()}")

Unique patients with CT rapport: 164


In [31]:
# add info
CT_rapport_lookup = (
    df_verr_rad_v.dropna(subset=["CODE_CODING_DISPLAY"])  # exclude missing before grouping
    .groupby(["patientPseuID", "Index_dt"])["CODE_CODING_DISPLAY"]
    .apply(list)
    .reset_index()
    .rename(columns={"CODE_CODING_DISPLAY": "CT_rapport_info"})
)

before = df_filter.shape[0]
df_filter = df_filter.merge(CT_rapport_lookup, on=["patientPseuID", "Index_dt"], how="left")
print(f"Rows before: {before}, after: {df_filter.shape[0]}")
assert df_filter.shape[0] == before, "Merge changed row count — check for duplicate keys in ok_lookup"

Rows before: 985, after: 985


# check radiology orders

In [32]:
path_orders_rad = path_manager.get_file_path("ARGUS25", "20260803", "rad_orders_20260803.sas7bdat")
df_orders_rad = pyreadstat.read_sas7bdat(path_orders_rad)[0]
df_orders_rad["studyId_1011"] = df_orders_rad["studyId_1011"].astype(int).astype(str)
df_orders_rad["Index_dt"] = df_orders_rad["index_dt"].dt.date

In [33]:
# filter only on patients with prediction
df_orders_rad = df_orders_rad.merge(df_filter[["patientPseuID", "Index_dt"]], how="inner")

In [34]:
mask = df_orders_rad["code_coding_display_original"].isin(verr_list)
df_orders_rad = df_orders_rad[mask]

In [35]:
mask = df_orders_rad["code_coding_display_original"].str.contains("MRI|MRA|PET|SPECT", case=False, na=False)
df_orders_rad_MRI_PET_SPECT = df_orders_rad[mask]
df_orders_rad_CT = df_orders_rad[~mask]

In [36]:
df_filter["MRI_order"] = np.nan
df_filter["MRI_order"] = np.where(
    df_filter.set_index(["patientPseuID", "Index_dt"]).index.isin(
        df_orders_rad_MRI_PET_SPECT.set_index(["patientPseuID", "Index_dt"]).index
    ),
    1,
    df_filter["MRI_order"],
)

print(f"Unique patients with MRI_order: {df_filter['patientPseuID'].loc[df_filter['MRI_order'] == 1].nunique()}")

Unique patients with MRI_order: 28


In [37]:
# add info
MRI_order_lookup = (
    df_orders_rad_MRI_PET_SPECT.dropna(subset=["code_coding_display_original"])  # exclude missing before grouping
    .groupby(["patientPseuID", "Index_dt"])["code_coding_display_original"]
    .apply(list)
    .reset_index()
    .rename(columns={"code_coding_display_original": "MRI_order_info"})
)

before = df_filter.shape[0]
df_filter = df_filter.merge(MRI_order_lookup, on=["patientPseuID", "Index_dt"], how="left")
print(f"Rows before: {before}, after: {df_filter.shape[0]}")
assert df_filter.shape[0] == before, "Merge changed row count — check for duplicate keys in ok_lookup"

Rows before: 985, after: 985


Everyone with MRI/PET/SPECT verrichting or Order should get label = 1. 

In [38]:
df_filter["CT_order"] = np.nan
df_filter["CT_order"] = np.where(
    df_filter.set_index(["patientPseuID", "Index_dt"]).index.isin(
        df_orders_rad_CT.set_index(["patientPseuID", "Index_dt"]).index
    ),
    1,
    df_filter["CT_order"],
)

print(f"Unique patients with CT_order: {df_filter['patientPseuID'].loc[df_filter['CT_order'] == 1].nunique()}")

Unique patients with CT_order: 15


In [39]:
# add info
CT_order_lookup = (
    df_orders_rad_CT.dropna(subset=["code_coding_display_original"])  # exclude missing before grouping
    .groupby(["patientPseuID", "Index_dt"])["code_coding_display_original"]
    .apply(list)
    .reset_index()
    .rename(columns={"code_coding_display_original": "CT_order_info"})
)

before = df_filter.shape[0]
df_filter = df_filter.merge(CT_order_lookup, on=["patientPseuID", "Index_dt"], how="left")
print(f"Rows before: {before}, after: {df_filter.shape[0]}")
assert df_filter.shape[0] == before, "Merge changed row count — check for duplicate keys in ok_lookup"


Rows before: 985, after: 985


In [40]:
# Add indicator column when CT or MRI is plannen for more than 180 days
waitinglist_threshold = 180

# Subset of radiology orders exceeding the threshold
df_orders_rad_long_waiting_list = df_orders_rad.loc[
    df_orders_rad["dif_index_date"] > waitinglist_threshold
].copy()

print(
    f"Radiology orders with long waiting list "
    f"(>{waitinglist_threshold} days): {df_orders_rad_long_waiting_list.shape}"
)

df_filter["Long_rad_waitinglist"] = np.nan

filter_index = df_filter.set_index(["patientPseuID", "Index_dt"]).index
long_wait_index = df_orders_rad_long_waiting_list.set_index(["patientPseuID", "Index_dt"]).index

# long wait if:
#   1. its (patientPseuID, Index_dt) matches a long-wait radiology order, AND
#   2. it is NOT already verified as MRI/PET/SPECT or CT
is_long_wait_match = filter_index.isin(long_wait_index)
is_verified_other = (df_filter["Ver_Rad_MRI_PET_SPECT"] == 1) | (df_filter["Ver_Rad_CT"] == 1)

df_filter["Long_rad_waitinglist"] = np.where(
    is_long_wait_match & ~is_verified_other,
    1,
    df_filter["Long_rad_waitinglist"],
)

# Count unique patients flagged with a long radiology waiting list
n_unique_patients = df_filter.loc[df_filter["Long_rad_waitinglist"] == 1, "patientPseuID"].nunique()
print(f"Unique patients with long radiology waiting list: {n_unique_patients}")

# drop patients with only a regular check up (waitinglist > 180 days)
df_filter = df_filter.loc[df_filter["Long_rad_waitinglist"] != 1]

Radiology orders with long waiting list (>180 days): (29, 16)
Unique patients with long radiology waiting list: 17


# Load CT rapports

### Load labels Bram

In [41]:
path_labels = path_manager.get_file_path("ARGUS25", "radio_labels", "silent_trial_labels", "labels_from_model.parquet")

df_labels = pd.read_parquet(path_labels)
df_labels["studyId_0888"] = df_labels["studyId_0888"].astype(int).astype(str)

In [42]:
df_labels = df_labels.merge(
    df_verr_rad_v[["patientPseuID", "index_dt", "CODE_CODING_DISPLAY"]], on=["patientPseuID", "index_dt"], how="left"
)

In [43]:
df_labels = df_labels.drop_duplicates()

In [44]:
# in case of multiple report rows, keep the max vlaues per column for one patient-index combination
def combine_labels(duplicate_group):  # noqa: D103
    non_null = duplicate_group.dropna()
    if len(non_null) == 0:
        return np.nan
    return non_null.max()


df_labels_clean = (
    df_labels.groupby(["patientPseuID", "index_dt"])
    .agg(
        {
            "text_model_proba": combine_labels,
            "macro_stenosis": combine_labels,
            "macro_plaque": combine_labels,
            "cacs_mean": combine_labels,
            "cacs_percentile_2001": combine_labels,
            "cacs_percentile_2020": combine_labels,
            "macro_stenosegraad": combine_labels,
            "macro_cadrads": combine_labels,
            "macro_mesa": combine_labels,
            "CODE_CODING_DISPLAY": combine_labels,
            "feature": combine_labels,
        }
    )
    .reset_index()
)

print(f"Original: {len(df_labels)} rows")
print(f"After deduplication: {len(df_labels_clean)} rows")

duplicated = df_labels_clean[df_labels_clean.duplicated(subset=["patientPseuID", "index_dt"], keep=False)]
print(f"Number of duplicated rows (based on study ID and index dt): {len(duplicated)}")

df_labels = df_labels_clean.copy()

Original: 1681 rows
After deduplication: 1223 rows
Number of duplicated rows (based on study ID and index dt): 0


In [45]:
# Keep only rows where CODE_CODING_DISPLAY is present
drop_Code_display = False

if drop_Code_display:
    df_labels = df_labels.loc[df_labels["CODE_CODING_DISPLAY"].notna()].copy()
    print(f"df_labels shape after dropping missing CODE_CODING_DISPLAY: {df_labels.shape}")

    print(df_labels["CODE_CODING_DISPLAY"].value_counts(dropna=False))

# Convert index_dt to date-only, stored as Index_dt to match df_filter's column naming
df_labels["Index_dt"] = df_labels["index_dt"].dt.date


In [46]:
# Initialize the flag column as NaN (no value assigned yet)
df_filter["CT_Bram"] = np.nan

# Build comparable (patientPseuID, Index_dt) indices for both dataframes
filter_index = df_filter.set_index(["patientPseuID", "Index_dt"]).index
labels_index = df_labels.set_index(["patientPseuID", "Index_dt"]).index

# Set flag = 1 where the patient/date combo has a matching label row,
# otherwise keep the existing (NaN) value
df_filter["CT_Bram"] = np.where(
    filter_index.isin(labels_index),
    1,
    df_filter["CT_Bram"],
)

# Count unique patients flagged with CT_Bram
n_unique_patients = df_filter.loc[df_filter["CT_Bram"] == 1, "patientPseuID"].nunique()
print(f"Unique patients with CT_Bram: {n_unique_patients}")


Unique patients with CT_Bram: 277


# Create label logic

In [47]:
# Label = 1 if any of these indicate a CAD-related order/referral happened
label_cols = ["Order_Ok", "Verrichting_Ok", "Ver_Rad_MRI_PET_SPECT", "MRI_order"]
mask_positive = (df_filter[label_cols] == 1).any(axis=1)
df_filter["Label"] = np.where(mask_positive, 1, np.nan)
print(f"Rows labeled 1 from order/referral columns: {int(df_filter['Label'].sum())}")

# Label = 0 if there is no information at all across any relevant column
cols_check = [
    "Order_Ok",
    "Verrichting_Ok",
    "Ver_Rad_MRI_PET_SPECT",
    "Ver_Rad_CT",
    "CT_rapport",
    "MRI_order",
    "CT_order",
    "CT_Bram",
]
mask_all_nan = df_filter[cols_check].isna().all(axis=1)
df_filter.loc[mask_all_nan, "Label"] = 0
print(df_filter["Label"].value_counts(dropna=False))


Rows labeled 1 from order/referral columns: 264
Label
0.0    569
1.0    264
NaN    135
Name: count, dtype: int64


In [48]:
print(f"Rows with Label still NaN: {df_filter.loc[df_filter['Label'].isna()].shape}")

# Among unlabeled rows, how many have a CT order verified?
subset = df_filter.loc[df_filter["Label"].isna() & (df_filter["Ver_Rad_CT"] == 1)]
print(f"Unlabeled rows with Ver_Rad_CT == 1: {subset.shape}")
print(subset["CT_Bram"].value_counts(dropna=False))

Rows with Label still NaN: (135, 60)
Unlabeled rows with Ver_Rad_CT == 1: (128, 60)
CT_Bram
1.0    122
NaN      6
Name: count, dtype: int64


In [49]:
# check how many records need a CT rapport but have none, these should be excluded:
# every row with label NaN by now whould have a CT outcome
df_filter.loc[df_filter["CT_Bram"].isna() & df_filter["Label"].isna()].shape

(11, 60)

In [50]:
# Exclude rows where we have neither a CT_Bram result nor any other basis for a Label
before = df_filter.shape
df_filter = df_filter.loc[~(df_filter["CT_Bram"].isna() & df_filter["Label"].isna())]
print(f"Dropped {before[0] - df_filter.shape[0]} rows with no CT_Bram and no Label. New shape: {df_filter.shape}")

Dropped 11 rows with no CT_Bram and no Label. New shape: (957, 60)


Rows dropped because they should have a CT but we do not know the outcome. Hence, we cannot verify the true label. 

## Extract labels from Bram

In [51]:
ct_cols = ["macro_stenosis", "macro_stenosegraad", "macro_cadrads", "cacs_mean", "macro_mesa"]
has_ct_info = df_labels[ct_cols].notna().any(axis=1)

df_labels["CT_based_CAD"] = (
    (
        (df_labels["macro_stenosis"] == 1)
        | (df_labels["macro_stenosegraad"] == 1)
        | (df_labels["macro_cadrads"] == 1)
        | (df_labels["cacs_mean"] > 400)
        | (df_labels["macro_mesa"] == 1)
    )
    .astype(int)
    .where(has_ct_info)
)

df_labels["cacs_zero"] = (df_labels["cacs_mean"] == 0).astype(int).where(df_labels["cacs_mean"].notna())

In [52]:
merge_cols = [
    "patientPseuID",
    "Index_dt",
    "CT_based_CAD",
    "cacs_zero",
    "cacs_mean",
    "macro_stenosis",
    "macro_stenosegraad",
    "macro_cadrads",
    "macro_mesa",
    "text_model_proba",
]

before_rows = df_filter.shape[0]
df_filter = df_filter.merge(
    df_labels[merge_cols],
    on=["patientPseuID", "Index_dt"],
    how="left",
)
print(f"Rows before merge: {before_rows}, after merge: {df_filter.shape[0]}")

Rows before merge: 957, after merge: 957


In [53]:
# Fill Label = 1 where CT report shows CAD
mask = df_filter["Label"].isna() & (df_filter["CT_based_CAD"] == 1)
print(f"Rows filled Label=1 from CT_based_CAD: {mask.sum()}")
df_filter.loc[mask, "Label"] = 1

# Fill Label = 0 where CACS = 0 (no calcified plaque)
mask = df_filter["Label"].isna() & (df_filter["cacs_zero"] == 1)
print(f"Rows filled Label=0 from cacs_zero: {mask.sum()}")
df_filter.loc[mask, "Label"] = 0

# Final check: how many rows remain unlabeled?
print(df_filter["Label"].value_counts(dropna=False))

Rows filled Label=1 from CT_based_CAD: 47
Rows filled Label=0 from cacs_zero: 9
Label
0.0    578
1.0    311
NaN     68
Name: count, dtype: int64


Drop inconclusive records

In [54]:
df_filter = df_filter.loc[df_filter["Label"].notna()]

In [55]:
print(df_filter["patientPseuID"].nunique())
print(
    f"Number of unique patients with CAD related symptoms, physical first consultation and less than 80% missingness: {df_filter['patientPseuID'].nunique()}"
)
print(df_filter.shape)
print(df_filter.groupby("gender")["Age"].describe())

print(df_filter.shape[0])
print(len(df_filter.loc[df_filter["gender"] == "1"]))
print(f"Percentage female: {100 / df_filter.shape[0] * len(df_filter.loc[df_filter['gender'] == '1'])}")

882
Number of unique patients with CAD related symptoms, physical first consultation and less than 80% missingness: 882
(889, 68)
        count       mean        std   min   25%   50%   75%   max
gender                                                           
0       445.0  60.991011  16.737639  18.0  53.0  63.0  74.0  92.0
1       444.0  59.238739  16.969678  18.0  50.0  62.0  73.0  92.0
889
444
Percentage female: 49.9437570303712


In [56]:
df_labels_final = df_filter.copy()

In [57]:
df_labels_final.shape

(889, 68)

In [58]:
print(df_labels_final["Label"].value_counts())
# as proportions (0-1)
print(df_labels_final["Label"].value_counts(normalize=True))

# as a percentage, just for the positive class (assuming 1 = positive)
pct_positive = (df_labels_final["Label"] == 1).mean() * 100
print(f"Positive rate: {pct_positive:.2f}%")

Label
0.0    578
1.0    311
Name: count, dtype: int64
Label
0.0    0.650169
1.0    0.349831
Name: proportion, dtype: float64
Positive rate: 34.98%


## add prediction label

In [59]:
df_labels_final["pred_label"] = (df_labels_final["predicted_risk"] >= 0.15).astype(int)
df_labels_final.shape

(889, 69)

In [60]:
df_labels_final["pred_label"].value_counts()

pred_label
0    601
1    288
Name: count, dtype: int64

In [61]:
## Save labels!
df_labels_final.to_parquet("../data/df_labels_final_12082026_info.parquet")